In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
from transformers import TFXLMRobertaModel, XLMRobertaTokenizer, XLMRobertaConfig
import matplotlib.pyplot as plt

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
df_train_original = pd.read_pickle("Data/df_preprocessed_train_text.pickle").reset_index(drop = True)
df_test_original = pd.read_pickle("Data/df_preprocessed_test_text.pickle").reset_index(drop = True)

In [4]:
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme"]
text_embedding_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
categorical_columns_original = base_n_encoder_cols + one_hot_columns

In [5]:
def filter_outliers(df, target_col, upper_limit, lower_limit):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = int(lower_limit * len(data_array))
    cut_off_high_indice = int(upper_limit * len(data_array))
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped".format(len(outlier_indices)))
    df.drop(labels = outlier_indices)
    return df

def get_train_test(df_train, df_test):
    #initialize the numerical and categorical data
    drop_columns = ["date_conclusion_contract"] + text_embedding_columns
    #get the training data
    X_train = df_train[[col for col in df_train.columns if col != "award_contract/val_total: 0" and col not in drop_columns]].to_numpy()
    y_train = df_train["award_contract/val_total: 0"].to_numpy().reshape(-1,1)
    #get the testing data
    X_test = df_test[[col for col in df_train.columns if col != "award_contract/val_total: 0" and col not in drop_columns]].to_numpy()
    y_test = df_test["award_contract/val_total: 0"].to_numpy().reshape(-1,1)

    return X_train, y_train, X_test, y_test

def get_text_data(df_train, df_test):
    #get the text data in a usable format
    X_train_text = []
    for i in df_train.index:
        text = ""
        for col in text_embedding_columns:
            text += str(df_train[col].loc[i])
            
        X_train_text.append(text)

    X_test_text = []
    for i in df_test.index:
        text = ""
        for col in text_embedding_columns:
            text += str(df_test[col].loc[i])
        
        X_test_text.append(text)
    
    return X_train_text, X_test_text

#initialize tokenizer and model
model_name = "jplu/tf-xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
#xlm_roberta_model = TFXLMRobertaModel.from_pretrained(model_name)

config = XLMRobertaConfig.from_pretrained(model_name)
config.hidden_size = 252  # You can adjust this value to make the hidden state smaller
xlm_roberta_model = TFXLMRobertaModel(config=config)


def create_xlm_roberta_embeddings(texts, batch_size = 32, max_length = 128):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        # Tokenize the batch of data
        tokenized_text = tokenizer(batch, return_tensors="tf", padding='max_length', truncation=True, max_length=max_length)
        input_ids = tokenized_text["input_ids"]
        attention_mask = tokenized_text["attention_mask"]

        # Connect the model to the inputs with the tensors
        output = xlm_roberta_model(input_ids=input_ids, attention_mask=attention_mask)

        # Extract embeddings
        batch_embeddings = output.last_hidden_state
        embeddings.append(batch_embeddings)
        embeddings = [tf.cast(embedding, dtype=tf.uint64) for embedding in embeddings]
    return tf.concat(embeddings, axis=0)

In [6]:
def old_not_working():
    #let's look at the characteristics of the text
    total = 0
    data_array = np.array([])
    for row_text in X_train_text:
        amount_of_words = len(row_text.split(" "))
        total += amount_of_words
        data_array = np.append(data_array, amount_of_words)

    mean = np.mean(data_array)
    std = np.std(data_array)
    min_value = np.min(data_array)
    max_value = np.max(data_array)
    print("mean: {}".format(mean), "\n\n",
          "std: {}".format(std), "\n\n",
          "min_value: {}".format(min_value), "\n\n",
          "max_value: {}".format(max_value))

    upper_bound = 1
    lower_bound = 0.00
    upper_bound_dict = {}

    lower_bound_indice = int(lower_bound * len(data_array))
    upper_bound_indice = int(upper_bound * len(data_array))
    data_array = np.sort(data_array)[lower_bound_indice:upper_bound_indice]

    #let's make a boxplot
    plt.boxplot(data_array)
    plt.title("Distribution of the amount of words")
    plt.show()

results from the whole dataset: <br>
mean: 47.37139295581352 <br>
std: 101.57871040293878 <br>
min_value: 1.0 <br>
max_value: 4613.0 <br> <br>
Although unlikely to be normally distirbuted, the differences in the plots show that by simply removing 5% of the outliers, the dataset looks drastically different with almost everything being below 120 tokens. As such, an embedding length of 128 is chosen which should be sufficient

In [7]:
df_train = copy.deepcopy(df_train_original)
df_test = copy.deepcopy(df_test_original)

In [ ]:
#import and concat word vectors
import os

# Specify the directory path
directory_path = "/path/to/your/directory"  # Replace with the actual directory path

# Get a list of filenames in the directory
file_names = os.listdir(directory_path)

# Print the file names
for file_name in file_names:
    print(file_name)

In [11]:
#df_train = filter_outliers(df_train, target_col="award_contract/val_total: 0", upper_limit = 1, lower_limit = 0.00)
X_train, y_train, X_test, y_test = get_train_test(df_train, df_test)
X_train_text, X_test_text = get_text_data(df_train, df_test)
X_train_embedded = create_xlm_roberta_embeddings(X_train_text)
#X_test_embedded = create_xlm_roberta_embeddings(X_test_text)

In [13]:
with open("Data/X_train_embedded_160000_166861", "wb") as f:
    pickle.dump(X_train_embedded, f)

#with open("Data/X_test_embedded_50000_70000", "wb") as f:
#    pickle.dump(X_test_embedded, f)

In [ ]:
with open("Data/X_train_embedded_10000", "rb") as f:
    X_train_embedded = pickle.load(f)

with open("Data/X_test_embedded_10000", "rb") as f:
    X_test_embedded = pickle.load(f)

In [ ]:
#define variables
LSTM_dropout = 0.2
numer_of_features = 128
len_feature_vec = 252
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_function = "mean_absolute_error"
#define input layer
input_layer_lstm = Input(shape=(numer_of_features,len_feature_vec), name="xlm_roberta_embedding")
lstm_layer_1 = LSTM(units=128, dropout= LSTM_dropout, return_sequences=True, activation="tanh")(input_layer_lstm)
lstm_layer_2 = LSTM(units=64, dropout= LSTM_dropout, return_sequences=True, activation="tanh")(lstm_layer_1)
lstm_layer_3 = LSTM(units=32, dropout= LSTM_dropout, return_sequences=True, activation="tanh")(lstm_layer_2)
lstm_layer_4 = LSTM(units=16, dropout= LSTM_dropout, return_sequences=True, activation="tanh")(lstm_layer_3)
lstm_layer_5 = LSTM(units=4, dropout= LSTM_dropout, return_sequences=False, activation="tanh")(lstm_layer_4)
# Add a Flatten layer before the Dense layer
#dense_layer = Dense(1)(lstm_layer_5)
#model_text = Model(input_layer_lstm, dense_layer)
#model_text.summary()

#model_text.compile(loss=loss_function, optimizer=optimizer)
#model_text.fit(
#    x=[X_train_embedded], y=y_train,
#    validation_data=([X_test_embedded], y_test),
#    epochs=100, batch_size = 32)

In [ ]:
#define variables
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_function = "mean_absolute_error"

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer_1 = Dense(64, activation = "relu")(input_num_cat)
num_cat_layer_2 = Dense(32, activation="relu")(num_cat_layer_1)
num_cat_layer_3 = Dense(16, activation="relu")(num_cat_layer_2)
num_cat_layer_4 = Dense(8, activation="relu")(num_cat_layer_3)
num_cat_layer_5 = Dense(4, activation="relu")(num_cat_layer_4)
#num_cat_layer_6 = Dense(1, activation="linear")(num_cat_layer_5)
#model_num_cat = Model(input_num_cat, num_cat_layer_6)
#model_num_cat.summary()

#model_num_cat.compile(loss=loss_function, optimizer=optimizer)
#model_num_cat.fit(
#    x=[X_train], y=y_train,
#    validation_data=([X_test], y_test),
#    epochs=100, batch_size = 32)

In [ ]:
#combined output
combined_input = Concatenate()([lstm_layer_5, num_cat_layer_5])
combined_layer = Dense(4, activation="relu")(combined_input)
final_layer = Dense(1, activation="linear")(combined_layer)
final_model = Model(inputs=[input_layer_lstm, input_num_cat],
                       outputs=final_layer)

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_function = "mean_absolute_error"
final_model.compile(loss=loss_function, optimizer=optimizer)
final_model.fit(
    x=[X_train_embedded, X_train], y=y_train,
    validation_data=([X_test_embedded, X_test], y_test),
    epochs=100, batch_size = 32)